# 🩺 Prédiction du Diabète — Régression Logistique
> Exercices 1 à 6 — Pipeline complet de classification binaire

## ⚙️ Installation & Imports

In [ ]:
# !pip install scikit-learn pandas numpy matplotlib seaborn  # décommenter si nécessaire

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings, io, zipfile, urllib.request

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score, ConfusionMatrixDisplay
)
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")

COLORS = {"pos": "#E74C3C", "neg": "#3498DB", "main": "#2C3E50",
          "light": "#ECF0F1", "accent": "#27AE60", "warn": "#F39C12"}
print("✔  Imports OK")

---
## 🌟 Exercice 1 — Compréhension du problème & collecte de données
- Charger le dataset diabète et l'explorer
- Compter les cas positifs / négatifs
- Diviser en ensembles entraînement / test

In [ ]:
# ── Chargement ───────────────────────────────────────────────────────────────
URL = ("https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/"
       "heads/main/Week%204/Day%202/Diabetes%20prediction%20dataset.zip")

try:
    with urllib.request.urlopen(URL) as resp:
        raw = resp.read()
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        csv_name = [n for n in z.namelist() if n.endswith(".csv")][0]
        with z.open(csv_name) as f:
            df = pd.read_csv(f)
    print(f"✔  Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
except Exception as e:
    print(f"⚠  Téléchargement impossible ({e}).\n   → Dataset synthétique généré.")
    np.random.seed(42)
    n = 100_000
    df = pd.DataFrame({
        "gender":              np.random.choice(["Male","Female","Other"], n, p=[.49,.50,.01]),
        "age":                 np.random.uniform(0, 80, n),
        "hypertension":        np.random.randint(0, 2, n),
        "heart_disease":       np.random.randint(0, 2, n),
        "smoking_history":     np.random.choice(["never","former","current","No Info"], n),
        "bmi":                 np.random.uniform(10, 60, n),
        "HbA1c_level":         np.random.uniform(3.5, 9.0, n),
        "blood_glucose_level": np.random.randint(80, 300, n),
        "diabetes":            np.random.randint(0, 2, n),
    })

# ── Exploration ───────────────────────────────────────────────────────────────
print("\n── Aperçu ───────────────────────────────────────────────────────")
display(df.head())
print("\n── Statistiques descriptives ────────────────────────────────────")
display(df.describe())
print("\n── Types & valeurs manquantes ───────────────────────────────────")
display(pd.DataFrame({"dtype": df.dtypes,
                      "nulls": df.isnull().sum(),
                      "null_%": (df.isnull().mean()*100).round(2)}))

# ── Comptage ──────────────────────────────────────────────────────────────────
pos = (df["diabetes"] == 1).sum()
neg = (df["diabetes"] == 0).sum()
print(f"\n── Distribution de la cible ─────────────────────────────────────")
print(f"   Cas positifs (diabète=1) : {pos:>7,}  ({pos/len(df)*100:.1f} %)")
print(f"   Cas négatifs (diabète=0) : {neg:>7,}  ({neg/len(df)*100:.1f} %)")

In [ ]:
# ── Visualisation distribution ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("Exercice 1 — Distribution de la variable cible", fontsize=14, fontweight="bold")

axes[0].bar(["Négatif (0)", "Positif (1)"], [neg, pos],
            color=[COLORS["neg"], COLORS["pos"]], edgecolor="white", linewidth=1.5)
for bar, val in zip(axes[0].patches, [neg, pos]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + len(df)*0.002,
                 f"{val:,}", ha="center", va="bottom", fontweight="bold", fontsize=11)
axes[0].set_title("Nombre de cas", fontsize=12)
axes[0].set_ylabel("Nombre d'individus")
axes[0].set_ylim(0, max(neg, pos) * 1.12)
axes[0].spines[["top","right"]].set_visible(False)

axes[1].pie([neg, pos], labels=["Négatif (0)", "Positif (1)"],
            colors=[COLORS["neg"], COLORS["pos"]],
            autopct="%1.1f%%", startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[1].set_title("Proportion", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Encodage & Split ──────────────────────────────────────────────────────────
df_model = df.copy()
for col in df_model.select_dtypes("object").columns:
    df_model[col] = LabelEncoder().fit_transform(df_model[col].astype(str))

X = df_model.drop("diabetes", axis=1)
y = df_model["diabetes"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

print(f"✔  Split effectué  —  Entraînement : {X_train.shape[0]:,}  |  Test : {X_test.shape[0]:,}")

---
## 🌟 Exercice 2 — Sélection & standardisation du modèle

In [ ]:
print("""
Choix du modèle : Régression Logistique
════════════════════════════════════════
• Problème de classification BINAIRE (diabète : oui / non).
• Interprétable : les coefficients montrent l'influence de chaque feature.
• Probabiliste : sortie ∈ [0,1] via la fonction sigmoïde → seuil ajustable.
• Efficace sur de grands datasets tabulaires avec peu de paramètres.
• Régularisation L2 intégrée (paramètre C) → limite le sur-apprentissage.

Normalisation requise ? OUI
═══════════════════════════
Les features ont des échelles très différentes :
  âge ≈ 0–80  |  glycémie ≈ 80–300  |  BMI ≈ 10–60  |  HbA1c ≈ 3.5–9
StandardScaler centre (μ=0) et réduit (σ=1) chaque variable pour
éviter qu'une feature domine l'optimisation du gradient (descente de gradient).
""")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit + transform sur train uniquement
X_test_sc  = scaler.transform(X_test)        # transform seulement sur test
print("✔  StandardScaler appliqué")

In [ ]:
# ── Avant / Après standardisation ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle("Exercice 2 — Standardisation des features", fontsize=14, fontweight="bold")

pd.DataFrame(X_train, columns=X.columns).iloc[:, :5].plot(
    kind="box", ax=axes[0], patch_artist=True,
    boxprops=dict(facecolor=COLORS["neg"], alpha=.6))
axes[0].set_title("Avant StandardScaler", fontsize=12)
axes[0].set_ylabel("Valeur brute")
axes[0].tick_params(axis="x", rotation=30)

pd.DataFrame(X_train_sc, columns=X.columns).iloc[:, :5].plot(
    kind="box", ax=axes[1], patch_artist=True,
    boxprops=dict(facecolor=COLORS["accent"], alpha=.6))
axes[1].set_title("Après StandardScaler", fontsize=12)
axes[1].set_ylabel("Valeur standardisée")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

---
## 🌟 Exercice 3 — Entraînement du modèle

In [ ]:
# ── Entraînement ─────────────────────────────────────────────────────────────
model = LogisticRegression(max_iter=1000, random_state=42, solver="lbfgs")
model.fit(X_train_sc, y_train)
print("✔  Modèle entraîné avec succès")

y_pred      = model.predict(X_test_sc)
y_pred_prob = model.predict_proba(X_test_sc)[:, 1]

# ── Importance des features (coefficients) ───────────────────────────────────
coef_df = (pd.DataFrame({"Feature": X.columns, "Coefficient": model.coef_[0]})
             .sort_values("Coefficient", ascending=False))

fig, ax = plt.subplots(figsize=(9, 4))
bar_colors = [COLORS["pos"] if v > 0 else COLORS["neg"] for v in coef_df["Coefficient"]]
ax.barh(coef_df["Feature"], coef_df["Coefficient"],
        color=bar_colors, edgecolor="white", linewidth=1.2)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Coefficients — Régression Logistique\n"
             "(rouge = facteur de risque  |  bleu = facteur protecteur)",
             fontsize=11, fontweight="bold")
ax.set_xlabel("Valeur du coefficient")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.show()

---
## 🌟 Exercice 4 — Métriques d'évaluation

In [ ]:
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
cm   = confusion_matrix(y_test, y_pred)

print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f} %)")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1-Score  : {f1:.4f}\n")
print(classification_report(y_test, y_pred, target_names=["Négatif","Positif"]))

In [ ]:
# ── 4a : Accuracy ─────────────────────────────────────────────────────────────
train_acc = accuracy_score(y_train, model.predict(X_train_sc))
fig, ax = plt.subplots(figsize=(6, 4.5))
bars = ax.bar(["Train Accuracy", "Test Accuracy"], [train_acc, acc],
              color=[COLORS["neg"], COLORS["pos"]], edgecolor="white", linewidth=1.5, width=0.4)
for bar, val in zip(bars, [train_acc, acc]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{val:.3f}", ha="center", fontweight="bold", fontsize=12)
ax.set_ylim(0, 1.12)
ax.axhline(0.8, color="grey", linestyle="--", linewidth=1, alpha=.6)
ax.text(1.22, 0.81, "seuil 80 %", fontsize=9, color="grey")
ax.set_title("Score de précision (Accuracy)", fontsize=13, fontweight="bold")
ax.set_ylabel("Score")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.show()

print(f"""
Commentaire Accuracy :
  • Train ({train_acc*100:.1f} %) ≈ Test ({acc*100:.1f} %) → pas de sur-apprentissage.
  • Un score > 80 % indique un modèle globalement fiable.""")

In [ ]:
# ── 4b : Matrice de confusion ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm,
                       display_labels=["Négatif (0)", "Positif (1)"]).plot(
    ax=ax, colorbar=True, cmap="Blues")
ax.set_title("Matrice de confusion", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"""
Commentaire :
  TN={tn:,}  correctement prédits sains
  TP={tp:,}  correctement prédits diabétiques
  FP={fp:,}  sains prédits diabétiques  (fausse alarme)
  FN={fn:,}  diabétiques prédits sains  ⚠️ risque médical""")

In [ ]:
# ── 4c : Precision / Recall / F1 ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4.5))
metrics_vals = {"Précision": prec, "Rappel": rec, "F1-Score": f1}
b = ax.bar(metrics_vals.keys(), metrics_vals.values(),
           color=[COLORS["neg"], COLORS["warn"], COLORS["accent"]],
           edgecolor="white", linewidth=1.5, width=0.4)
for bar, val in zip(b, metrics_vals.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f"{val:.3f}", ha="center", fontweight="bold", fontsize=12)
ax.set_ylim(0, 1.12)
ax.set_title("Précision — Rappel — F1-Score", fontsize=13, fontweight="bold")
ax.set_ylabel("Score")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.show()

print(f"""
Commentaire :
  • Précision ({prec:.2f}) : {prec*100:.0f} % des individus prédits positifs sont réellement diabétiques.
  • Rappel    ({rec:.2f}) : {rec*100:.0f} % des vrais diabétiques sont détectés.
  • F1        ({f1:.2f}) : équilibre entre précision et rappel.
  → En santé, on priorise le RAPPEL pour minimiser les faux négatifs.""")

---
## 🌟 Exercice 5 — Frontière de décision (PCA 2D)

In [ ]:
# ── Réduction PCA → 2D ────────────────────────────────────────────────────────
pca    = PCA(n_components=2, random_state=42)
X_2d   = pca.fit_transform(X_test_sc)
var_ex = pca.explained_variance_ratio_

model_2d = LogisticRegression(max_iter=1000, random_state=42)
model_2d.fit(pca.transform(X_train_sc), y_train)
acc_2d = accuracy_score(y_test, model_2d.predict(X_2d))

# ── Grille ────────────────────────────────────────────────────────────────────
h = 0.04
xx, yy = np.meshgrid(np.arange(X_2d[:,0].min()-1, X_2d[:,0].max()+1, h),
                     np.arange(X_2d[:,1].min()-1, X_2d[:,1].max()+1, h))
Z = model_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

# ── Tracé ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
ax.contourf(xx, yy, Z, alpha=0.25, colors=[COLORS["neg"], COLORS["pos"]])
ax.contour(xx,  yy, Z, colors=COLORS["main"], linewidths=1.5)

idx = np.random.choice(len(X_2d), min(3000, len(X_2d)), replace=False)
sc  = ax.scatter(X_2d[idx,0], X_2d[idx,1],
                 c=y_test.values[idx], cmap=plt.cm.RdBu_r,
                 edgecolors="white", linewidths=0.4, s=20, alpha=0.75)
plt.colorbar(sc, ax=ax, label="Classe réelle  (0 = Négatif | 1 = Positif)")
ax.set_xlabel(f"PC1  ({var_ex[0]*100:.1f} % variance)", fontsize=11)
ax.set_ylabel(f"PC2  ({var_ex[1]*100:.1f} % variance)", fontsize=11)
ax.set_title(f"Frontière de décision — Régression Logistique (PCA 2D)\n"
             f"Accuracy 2D : {acc_2d*100:.2f} %  |  Variance expliquée : {var_ex.sum()*100:.1f} %",
             fontsize=13, fontweight="bold")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.show()

print(f"""
Commentaire :
  • La frontière est une droite car la régression logistique est un séparateur linéaire.
  • PCA conserve {var_ex.sum()*100:.1f} % de la variance — une perte d'information est inévitable.
  • L'accuracy 2D ({acc_2d*100:.2f} %) est inférieure au modèle complet car la projection
    réduit l'information disponible.""")

---
## 🌟 Exercice 6 — Courbe ROC
> Référence : https://www.statology.org/plot-roc-curve-python/

In [ ]:
# ── Calcul ────────────────────────────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
auc     = roc_auc_score(y_test, y_pred_prob)
opt_idx = np.argmax(tpr - fpr)          # point optimal (coin supérieur gauche)
opt_thr = thresholds[opt_idx]

# ── Tracé ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(fpr, tpr, color=COLORS["pos"], lw=2.5,
        label=f"Régression Logistique  (AUC = {auc:.4f})")
ax.fill_between(fpr, tpr, alpha=0.12, color=COLORS["pos"])
ax.plot([0,1], [0,1], "k--", lw=1.5, label="Classifieur aléatoire  (AUC = 0.50)")
ax.scatter(fpr[opt_idx], tpr[opt_idx], s=140, zorder=5,
           color=COLORS["accent"], edgecolors="white", linewidths=1.5,
           label=f"Seuil optimal = {opt_thr:.2f}")
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.05])
ax.set_xlabel("Taux de Faux Positifs — FPR (1 − Spécificité)", fontsize=12)
ax.set_ylabel("Taux de Vrais Positifs — TPR (Sensibilité / Recall)", fontsize=12)
ax.set_title(f"Courbe ROC — Régression Logistique\nAUC = {auc:.4f}",
             fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=11)
ax.spines[["top","right"]].set_visible(False)
ax.grid(alpha=.3)
plt.tight_layout()
plt.show()

print(f"""
Commentaire :
  • AUC = {auc:.4f}
    — > 0.90 : excellent  |  0.80–0.90 : bon  |  0.70–0.80 : acceptable  |  0.50 : aléatoire
  • Le point vert (seuil = {opt_thr:.2f}) maximise TPR − FPR (coin supérieur gauche).
  • En médecine, abaisser le seuil augmente le rappel au prix de plus de fausses alarmes.""")